# 03 - Segmentación de clientes fintech con KMeans

## Caso fintech
Este notebook agrupa clientes según su comportamiento financiero.

## Objetivo
Construir segmentos como clientes premium, clientes transaccionales, clientes endeudados o clientes inactivos.

## Dataset
Se genera un dataset sintético de 2,500 clientes.

## Técnicas
- Clustering no supervisado
- Estandarización
- Método del codo
- KMeans
- PCA para visualización


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

np.random.seed(42)


In [ ]:
n = 2500

monthly_income = np.random.normal(4500, 2200, n).clip(1000, 20000)
monthly_spend = (monthly_income * np.random.beta(2, 4, n)).clip(100, 18000)
avg_balance = np.random.lognormal(mean=7.4, sigma=0.9, size=n).clip(100, 50000)
num_transactions = np.random.poisson(35, n)
credit_card_utilization = np.random.beta(2.3, 3.2, n)
loan_balance = np.random.lognormal(mean=7.2, sigma=1.1, size=n).clip(0, 60000)
digital_sessions = np.random.poisson(18, n)
products_count = np.random.choice([1, 2, 3, 4, 5], n, p=[0.25, 0.3, 0.25, 0.15, 0.05])

df = pd.DataFrame({
    "monthly_income": monthly_income.round(2),
    "monthly_spend": monthly_spend.round(2),
    "avg_balance": avg_balance.round(2),
    "num_transactions": num_transactions,
    "credit_card_utilization": credit_card_utilization.round(3),
    "loan_balance": loan_balance.round(2),
    "digital_sessions": digital_sessions,
    "products_count": products_count
})

df.head()


In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
features = [
    "monthly_income", "monthly_spend", "avg_balance", "num_transactions",
    "credit_card_utilization", "loan_balance", "digital_sessions", "products_count"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])


In [ ]:
# Método del codo para elegir cantidad de clusters
inertias = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.plot(range(2, 11), inertias, marker="o")
plt.title("Método del codo")
plt.xlabel("Número de clusters")
plt.ylabel("Inertia")
plt.show()


In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

df["cluster"].value_counts().sort_index()


In [ ]:
cluster_summary = df.groupby("cluster")[features].mean().round(2)
cluster_summary

In [ ]:
# Visualización con PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "pc1": X_pca[:, 0],
    "pc2": X_pca[:, 1],
    "cluster": df["cluster"]
})

plt.figure(figsize=(8, 6))
for cluster in sorted(plot_df["cluster"].unique()):
    subset = plot_df[plot_df["cluster"] == cluster]
    plt.scatter(subset["pc1"], subset["pc2"], label=f"Cluster {cluster}", alpha=0.6)

plt.title("Segmentación de clientes con PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()


In [ ]:
# Etiquetado interpretativo simple
def describe_cluster(row):
    if row["monthly_income"] > cluster_summary["monthly_income"].mean() and row["avg_balance"] > cluster_summary["avg_balance"].mean():
        return "clientes_de_alto_valor"
    if row["loan_balance"] > cluster_summary["loan_balance"].mean() and row["credit_card_utilization"] > cluster_summary["credit_card_utilization"].mean():
        return "clientes_endeudados"
    if row["num_transactions"] > cluster_summary["num_transactions"].mean() and row["digital_sessions"] > cluster_summary["digital_sessions"].mean():
        return "clientes_digitales_activos"
    return "clientes_masivos"

cluster_profiles = cluster_summary.apply(describe_cluster, axis=1)
cluster_profiles
